<a href="https://colab.research.google.com/github/Miranics/TradesREvamp/blob/main/research/02_strategy_bakeoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Strategy bake-off: gold, 2024 to 2026

Tests six intraday gold strategies and two trend-following variants on the same 1-minute data, with the same costs, year by year. The sweep model was designed on 2026 data, so **2024 and 2025 are its out-of-sample test**.

Run the cells top to bottom (Runtime → Run all works). The first run downloads about 34 months of data into `My Trades/data` in your Drive; later runs reuse it. Copy everything printed from Step 6 onwards back to the chat.

| Code | Strategy | Source of the idea |
|---|---|---|
| A1, A2 | Sweep model: London range → NY sweep → 5-min displacement → FVG entry (hold, or with adds) | Your ICT model, tested in this project |
| B | NY intraday momentum: first 30 min direction, hold to 16:00 ET | Peer-reviewed intraday momentum studies |
| C | NY opening-range breakout: first 5-min candle, stop at its other side, 10R | Zarattini & Aziz, plus an independent replication |
| D1, D2 | Session drift: always sell NY hours / always buy Asian hours | World Gold Council 2026 session data |
| E | Trend following on daily closes, 20 and 60 days | Time-series momentum research |

**Step 1.** Connect Google Drive and find the `My Trades` folder (approve the Drive pop-up).

In [ ]:
import glob
import os

from google.colab import drive

drive.mount("/content/drive")

matches = [p for p in glob.glob("/content/drive/MyDrive/*") if os.path.basename(p).strip() == "My Trades"]
if not matches:
    raise FileNotFoundError("Could not find a folder called 'My Trades' at the top of My Drive")
BASE = matches[0]
DATA = os.path.join(BASE, "data")
OUT = os.path.join(BASE, "results")
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("Data folder:   ", DATA)
print("Results folder:", OUT)


**Step 2.** Download 1-minute gold, one month at a time, into Drive. Takes about 15–25 minutes the first time. If some months fail, wait a few minutes and run it again; it only fetches what's missing.

In [ ]:
import subprocess
import time
from datetime import date

START, END = date(2024, 1, 1), date(2026, 10, 10)


def month_ranges(start, end):
    y, m = start.year, start.month
    while date(y, m, 1) < end:
        ny, nm = (y + 1, 1) if m == 12 else (y, m + 1)
        yield date(y, m, 1), min(date(ny, nm, 1), end)
        y, m = ny, nm


def download(instrument, prefix):
    saved, skipped, failed = 0, 0, []
    for a, b in month_ranges(START, END):
        name = f"{prefix}_{a:%Y_%m}"
        path = os.path.join(DATA, name + ".csv")
        if os.path.exists(path) and os.path.getsize(path) > 100_000 and b.day == 1:
            skipped += 1
            continue
        cmd = ["npx", "--yes", "dukascopy-node", "-i", instrument, "-from", str(a), "-to", str(b),
               "-t", "m1", "-f", "csv", "-dir", DATA, "-fn", name,
               "-bs", "2", "-bp", "3000", "-r", "10", "-rp", "15000", "-s"]
        out = subprocess.run(cmd, capture_output=True, text=True).stdout
        if "File saved" in out:
            saved += 1
            print(f"{name}: saved")
        else:
            failed.append(name)
            print(f"{name}: FAILED")
            if os.path.exists(path):
                os.remove(path)
        time.sleep(5)
    print(f"\n{instrument}: {saved} downloaded, {skipped} already in Drive, {len(failed)} failed")
    if failed:
        print("Wait a few minutes and run this cell again. It only retries the missing months.")


download("xauusd", "xauusd")


**Step 3.** Load the gold data and check for missing months.

In [ ]:
import glob
import os

import numpy as np
import pandas as pd

SPREAD = 0.30
SPREAD_HIGH = 0.60
OZ_PER_001_LOT = 1


def load_m1(pattern):
    files = sorted(glob.glob(os.path.join(DATA, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {DATA}")
    df = pd.concat(pd.read_csv(f) for f in files)
    df["time"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df.drop_duplicates("time").set_index("time").sort_index()
    df = df[["open", "high", "low", "close"]].astype(float)
    return df[df.index.dayofweek < 5]


m1 = load_m1("xauusd_*.csv")
m1_ny = m1.tz_convert("America/New_York")

per_month = m1.groupby(m1.index.strftime("%Y-%m")).size()
print(f"Gold minutes loaded: {len(m1):,}  from {m1.index[0]:%Y-%m-%d} to {m1.index[-1]:%Y-%m-%d}")
thin = per_month.iloc[:-1][per_month.iloc[:-1] < 15000]
if len(thin):
    print("Months with too little data (re-run Step 2 to fill them):", ", ".join(thin.index))
else:
    print("Every month looks complete.")


**Step 4.** Load the trading logic (prints nothing). The sweep model code is identical to the earlier tests.

In [ ]:
DISPLACEMENT = 1.5
BUFFER = 0.50


def bracket(d, entry, stop, path, rr=None):
    """Gross points for one position entered at `entry` on the first bar of `path`.
    Stop and target are checked on every bar; if both are touched in the same bar the stop wins."""
    risk = abs(entry - stop)
    if risk <= 0 or len(path) == 0:
        return None
    hi, lo = path["high"].to_numpy(), path["low"].to_numpy()
    n = len(path)
    sl = (lo <= stop) if d == 1 else (hi >= stop)
    i_sl = int(np.argmax(sl)) if sl.any() else n
    i_tp = n
    if rr is not None:
        target = entry + d * rr * risk
        tp = (hi >= target) if d == 1 else (lo <= target)
        i_tp = int(np.argmax(tp)) if tp.any() else n
    if i_tp < i_sl:
        return rr * risk
    if i_sl < n:
        return -risk
    return d * (path["close"].iloc[-1] - entry)


def swing_points(b, side):
    s = b["low"] if side == "low" else b["high"]
    if side == "low":
        return (s < s.shift(1)) & (s < s.shift(2)) & (s < s.shift(-1)) & (s < s.shift(-2))
    return (s > s.shift(1)) & (s > s.shift(2)) & (s > s.shift(-1)) & (s > s.shift(-2))


def find_setup(b, lh, ll):
    body = (b["close"] - b["open"]).abs()
    avg_body = body.rolling(20, min_periods=5).median().shift(1)
    ny = b[(b.index.hour >= 13) & (b.index.hour < 17)]
    for t in ny.index:
        i = b.index.get_loc(t)
        bar = b.iloc[i]
        for d, swept in ((-1, bar["high"] > lh), (1, bar["low"] < ll)):
            if not swept:
                continue
            prior = b.iloc[max(0, i - 40):i]
            sw = swing_points(b.iloc[max(0, i - 42):i + 1], "low" if d == -1 else "high")
            sw = sw.reindex(prior.index).fillna(False).astype(bool)
            if not sw.any():
                continue
            level = prior.loc[sw[sw].index[-1], "low" if d == -1 else "high"]
            extreme = bar["high"] if d == -1 else bar["low"]
            for j in range(i + 1, min(i + 30, len(b))):
                x = b.iloc[j]
                extreme = max(extreme, x["high"]) if d == -1 else min(extreme, x["low"])
                broke = x["close"] < level if d == -1 else x["close"] > level
                ab = avg_body.iloc[j]
                strong = body.iloc[j] >= DISPLACEMENT * (ab if ab and not np.isnan(ab) else np.inf)
                if broke and strong:
                    for k in range(j + 1, i, -1):
                        if k >= len(b) or k - 2 < 0:
                            continue
                        a, c = b.iloc[k - 2], b.iloc[k]
                        if d == -1 and a["low"] > c["high"]:
                            return dict(dir=d, entry=c["high"], stop=extreme + BUFFER,
                                        ready=b.index[min(k + 1, len(b) - 1)])
                        if d == 1 and a["high"] < c["low"]:
                            return dict(dir=d, entry=c["low"], stop=extreme - BUFFER,
                                        ready=b.index[min(k + 1, len(b) - 1)])
                    return None
            return None
    return None


def find_fill(s, d1):
    d, entry, stop = s["dir"], s["entry"], s["stop"]
    after = d1[(d1.index >= s["ready"]) & (d1.index.hour < 21)]
    for t, x in after.iterrows():
        if (d == -1 and x["high"] >= stop) or (d == 1 and x["low"] <= stop):
            return None
        if (d == -1 and x["high"] >= entry) or (d == 1 and x["low"] <= entry):
            return None if t.hour >= 20 else t
    return None


def run_from_fill(d, entry, stop, d1, fill_t, adds=()):
    """Gross points and number of positions for a trade filled at fill_t, managed to 21:00 UTC."""
    risk = abs(entry - stop)
    window = d1[(d1.index >= fill_t) & (d1.index.hour < 21)]
    after = window[window.index > fill_t]
    units, todo = [entry], list(adds)
    for t, x in after.iterrows():
        hit_sl = x["high"] >= stop if d == -1 else x["low"] <= stop
        if hit_sl:
            return sum(d * (stop - e) for e in units), len(units)
        best = (entry - x["low"]) if d == -1 else (x["high"] - entry)
        while todo and best >= todo[0] * risk:
            units.append(entry + d * todo.pop(0) * risk)
            stop = entry
    last = window["close"].iloc[-1]
    return sum(d * (last - e) for e in units), len(units)


**Step 5.** Run every strategy over every day (a few minutes). Prints the number of trades per strategy per year.

In [ ]:
rows = []


def add(strategy, day, d, gross, units, gross_opp, units_opp, risk=np.nan):
    if gross is None or gross_opp is None:
        return
    rows.append(dict(strategy=strategy, day=pd.Timestamp(day), dir=d, gross=gross, units=units,
                     gross_opp=gross_opp, units_opp=units_opp, risk=risk))


# A. Sweep model (London range -> NY sweep -> 5-min displacement break -> FVG limit entry)
for day, d1 in m1.groupby(m1.index.date):
    london = d1[(d1.index.hour >= 7) & (d1.index.hour < 12)]
    if len(london) < 200:
        continue
    b = d1.resample("5min").agg({"open": "first", "high": "max", "low": "min", "close": "last"}).dropna()
    s = find_setup(b, london["high"].max(), london["low"].min())
    if not s:
        continue
    fill_t = find_fill(s, d1)
    if fill_t is None:
        continue
    d, entry, stop = s["dir"], s["entry"], s["stop"]
    opp_stop = 2 * entry - stop
    risk = abs(entry - stop)
    for name, adds in (("A1 Sweep model, hold to 23:00", ()), ("A2 Sweep model, adds at +1R/+2R", (1, 2))):
        g, u = run_from_fill(d, entry, stop, d1, fill_t, adds)
        go, uo = run_from_fill(-d, entry, opp_stop, d1, fill_t, adds)
        add(name, day, d, g, u, go, uo, risk)

# B, C, D. New York session strategies (times in New York time, so they follow US daylight saving)
for day, d1 in m1_ny.groupby(m1_ny.index.date):
    hm = d1.index.hour * 60 + d1.index.minute
    first30 = d1[(hm >= 570) & (hm < 600)]
    first5 = d1[(hm >= 570) & (hm < 575)]
    session = d1[(hm >= 600) & (hm < 960)]
    after_orb = d1[(hm >= 575) & (hm < 960)]
    if len(first30) < 25 or len(session) < 300:
        continue

    # B. Intraday momentum: direction of 09:30-10:00, held 10:00-16:00, no stop
    move = first30["close"].iloc[-1] - first30["open"].iloc[0]
    if move != 0:
        d = 1 if move > 0 else -1
        entry, last = session["open"].iloc[0], session["close"].iloc[-1]
        add("B  NY momentum (first 30 min -> hold to 16:00 ET)", day, d,
            d * (last - entry), 1, -d * (last - entry), 1)

    # C. Opening range breakout: direction of the first 5-min candle, stop at its other side, 10R or 16:00 ET
    o, c = first5["open"].iloc[0], first5["close"].iloc[-1]
    hi, lo = first5["high"].max(), first5["low"].min()
    if c != o and len(after_orb):
        d = 1 if c > o else -1
        entry = after_orb["open"].iloc[0]
        stop, opp_stop = (lo, hi) if d == 1 else (hi, lo)
        g = bracket(d, entry, stop, after_orb, rr=10)
        go = bracket(-d, entry, opp_stop, after_orb, rr=10)
        add("C  NY opening-range breakout (5-min, 10R)", day, d, g, 1, go, 1, abs(entry - stop))

    # D. Session drift: always short New York hours 09:30-16:00 ET
    ny = d1[(hm >= 570) & (hm < 960)]
    add("D1 Always SELL New York hours", day, -1, ny["open"].iloc[0] - ny["close"].iloc[-1], 1,
        ny["close"].iloc[-1] - ny["open"].iloc[0], 1)

# D2. Session drift: always long Asian hours 00:00-07:00 UTC
for day, d1 in m1.groupby(m1.index.date):
    asia = d1[d1.index.hour < 7]
    if len(asia) < 300:
        continue
    add("D2 Always BUY Asian hours", day, 1, asia["close"].iloc[-1] - asia["open"].iloc[0], 1,
        asia["open"].iloc[0] - asia["close"].iloc[-1], 1)

trades = pd.DataFrame(rows)
trades["year"] = trades["day"].dt.year
print(trades.groupby(["strategy", "year"]).size().unstack(fill_value=0).to_string())


**Step 6.** Scoreboard by year: win rate, profit, profit factor, t-stat, and whether it beats random direction.

In [ ]:
rng = np.random.default_rng(42)
N_RANDOM = 5000


def stats(net, net_opp):
    n = len(net)
    if n == 0:
        return {}
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    sd = net.std(ddof=1) if n > 1 else np.nan
    pick = rng.random((N_RANDOM, n)) < 0.5
    rand_tot = np.where(pick, net.to_numpy(), net_opp.to_numpy()).sum(axis=1)
    return {"trades": n,
            "win_%": round(100 * (net > 0).mean(), 1),
            "avg_$": round(net.mean(), 2),
            "total_$": round(net.sum(), 1),
            "profit_factor": round(wins / losses, 2) if losses > 0 else np.inf,
            "t_stat": round(net.mean() / sd * np.sqrt(n), 2) if sd and sd > 0 else np.nan,
            "p_vs_random": round((rand_tot >= net.sum()).mean(), 3)}


def scoreboard(spread, by=("strategy", "year")):
    t = trades.copy()
    t["net"] = t["gross"] - t["units"] * spread
    t["net_opp"] = t["gross_opp"] - t["units_opp"] * spread
    out = []
    for key, g in t.groupby(list(by)):
        out.append({**dict(zip(by, key if isinstance(key, tuple) else (key,))), **stats(g["net"], g["net_opp"])})
    for key, g in t.groupby("strategy"):
        row = {"strategy": key, **{b: "ALL" for b in by if b != "strategy"}}
        out.append({**row, **stats(g["net"], g["net_opp"])})
    return pd.DataFrame(out).sort_values(list(by), key=lambda s: s.astype(str)).reset_index(drop=True)


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("Profit is in $ per 0.01 lot (1 oz of gold). 'p_vs_random' = share of 5,000 random-direction")
print("versions of the same trades that did at least as well. Below 0.05 means the direction choice adds real value.\n")
print(f"## Scoreboard by year (spread ${SPREAD:.2f} per position)\n")
board = scoreboard(SPREAD)
print(board.to_string(index=False))


**Step 7.** Buys vs sells, the sweep model in R, and the same results with a doubled spread.

In [ ]:
t = trades.copy()
t["net"] = t["gross"] - t["units"] * SPREAD
t["side"] = np.where(t["dir"] == 1, "buy", "sell")

print("## Buys vs sells, by year (total $ per 0.01 lot)\n")
two_sided = t[~t["strategy"].str.startswith("D")]
print(two_sided.pivot_table(index=["strategy", "side"], columns="year", values="net",
                            aggfunc="sum", margins=True, margins_name="ALL").round(1).to_string())

print("\n## Sweep model in R (1R = distance from entry to stop)\n")
sw = t[t["strategy"].str.startswith("A")].copy()
sw["R"] = sw["net"] / sw["risk"]
print(sw.groupby(["strategy", "year"]).agg(
    trades=("R", "size"), avg_R=("R", lambda x: round(x.mean(), 3)), total_R=("R", lambda x: round(x.sum(), 1)),
    median_stop_dollars=("risk", lambda x: round(x.median(), 2))).to_string())
print("\n2026 is the period the model was designed on. 2024 and 2025 are the true out-of-sample test.")

print(f"\n## Same scoreboard with a wider spread (${SPREAD_HIGH:.2f}), all years combined\n")
wide = scoreboard(SPREAD_HIGH)
print(wide[wide["year"] == "ALL"].drop(columns="year").to_string(index=False))


**Step 8.** Trend following on daily closes, compared with simply buying and holding.

In [ ]:
day_key = (m1_ny.index - pd.Timedelta(hours=17)).date
daily = m1_ny["close"].groupby(day_key).last()
daily.index = pd.to_datetime(daily.index)
daily = daily[daily.index.dayofweek < 5]
chg = daily.diff()

tsm_rows = []
for lb in (20, 60):
    pos = np.sign(daily - daily.shift(lb)).shift(1)
    flips = pos.diff().abs().fillna(0) / 2
    pnl = (pos * chg - flips * SPREAD).dropna()
    for yr, g in pnl.groupby(pnl.index.year):
        tsm_rows.append({"strategy": f"E  Trend following, {lb}-day (daily, multi-day holds)", "year": yr,
                         "days": len(g), "position_changes": int(flips.loc[g.index].sum()),
                         "total_$": round(g.sum(), 1),
                         "t_stat": round(g.mean() / g.std(ddof=1) * np.sqrt(len(g)), 2)})
for yr, g in chg.dropna().groupby(chg.dropna().index.year):
    tsm_rows.append({"strategy": "Buy and hold (reference)", "year": yr, "days": len(g), "position_changes": 0,
                     "total_$": round(g.sum(), 1), "t_stat": round(g.mean() / g.std(ddof=1) * np.sqrt(len(g)), 2)})

print("## Trend following on daily closes (17:00 New York), $ per 0.01 lot\n")
print(pd.DataFrame(tsm_rows).to_string(index=False))
print("\nOvernight swap is not included. On Exness gold it can add up over multi-day holds.")


**Step 9.** Verdict: which strategies pass all four robustness checks. Also saves the results to `My Trades/results`.

In [ ]:
verdict = []
for name, g in board[board["year"] != "ALL"].groupby("strategy"):
    allrow = board[(board["strategy"] == name) & (board["year"] == "ALL")].iloc[0]
    yearly = dict(zip(g["year"], g["total_$"]))
    every_year_positive = all(v > 0 for v in yearly.values())
    wide_total = wide[(wide["strategy"] == name) & (wide["year"] == "ALL")]["total_$"].iloc[0]
    checks = {"positive every year": every_year_positive,
              "t-stat above 2": allrow["t_stat"] > 2,
              "beats random direction (p < 0.05)": allrow["p_vs_random"] < 0.05,
              "still positive at wide spread": wide_total > 0}
    verdict.append(dict(strategy=name, **{str(y): v for y, v in yearly.items()},
                        checks_passed=f"{sum(checks.values())}/4",
                        failed=", ".join(k for k, v in checks.items() if not v) or "none"))

print("## Verdict: totals in $ per 0.01 lot, and which robustness checks each strategy passes\n")
print(pd.DataFrame(verdict).sort_values("checks_passed", ascending=False).to_string(index=False))
print("\nA strategy worth automating should pass all 4. D1 and D2 are fixed-direction rules, so")
print("'beats random direction' only means the market drifted that way in this sample.")

trades.to_csv(os.path.join(OUT, "bakeoff_trades.csv"), index=False)
board.to_csv(os.path.join(OUT, "bakeoff_scoreboard.csv"), index=False)
print(f"\nSaved bakeoff_trades.csv and bakeoff_scoreboard.csv to {OUT}")


## Optional: dollar regime test
**Step 10.** Download 1-minute dollar index data into Drive (same retry behaviour as Step 2).

In [ ]:
download("dollaridxusd", "dollaridx")


**Step 11.** Does the sweep model do better when the dollar has a clear direction before 15:00 Kigali?

In [ ]:
dxy = load_m1("dollaridx_*.csv")

dd = {}
for day, d in dxy.groupby(dxy.index.date):
    pre = d[d.index.hour < 13]
    if len(pre) < 300:
        continue
    dd[pd.Timestamp(day)] = dict(high=d["high"].max(), low=d["low"].min(), pre_high=pre["high"].max(),
                                 pre_low=pre["low"].min(), move=pre["close"].iloc[-1] - pre["open"].iloc[0])
dd = pd.DataFrame(dd).T.sort_index()
dd["prev_high"], dd["prev_low"] = dd["high"].shift(1), dd["low"].shift(1)
dd["strong"] = dd["move"].abs() > dd["move"].abs().rolling(40, min_periods=10).median().shift(1)
dd["breakout"] = (dd["pre_high"] > dd["prev_high"]) ^ (dd["pre_low"] < dd["prev_low"])

sw = trades[trades["strategy"].str.startswith("A")].copy()
sw["net"] = sw["gross"] - sw["units"] * SPREAD
sw["R"] = sw["net"] / sw["risk"]
sw = sw.join(dd[["move", "strong", "breakout"]], on="day").dropna(subset=["move"])
sw["dollar_breakout"] = sw["breakout"].astype(bool)
sw["dollar_strong_move"] = sw["strong"].astype(bool)
sw["gold_trade_against_dollar_move"] = sw["dir"] == -np.sign(sw["move"])
sw["against_dollar_and_strong"] = sw["gold_trade_against_dollar_move"] & sw["dollar_strong_move"]

print("Dollar signals are all measured before 15:00 Kigali (13:00 UTC), so they are known before the trade.\n")
for col in ["dollar_breakout", "dollar_strong_move", "gold_trade_against_dollar_move", "against_dollar_and_strong"]:
    print(f"## {col}\n")
    print(sw.groupby(["strategy", col]).agg(trades=("R", "size"), win_pct=("R", lambda x: round((x > 0).mean() * 100, 1)),
          avg_R=("R", lambda x: round(x.mean(), 3)), total_R=("R", lambda x: round(x.sum(), 1))).to_string(), "\n")
